# BMCS2203 Artificial Intelligence
## River Water Quality Prediction: Model Training And Evaluation
#### Group Members: Chang Han Yean (SVM), Elwin Goh Yao Zu (Random Forest), Kaizen Soh (Decision Tree)

This notebook trains and evaluates Random Forest, Decision Tree, and Support Vector Machine models using the preprocessed water-quality training and testing datasets.


# 1.0 Project Setup

This section imports the required machine-learning libraries and defines the file paths used in model training.


## 1.1 Import Required Libraries

The models are trained using scikit-learn. `joblib` is used to save the trained models.


In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

print("Libraries imported successfully.")


## 1.2 Define Paths And Constants

The 1,150 training records are used for model fitting and hyperparameter tuning. The 650 testing records are used only for final evaluation.


In [ ]:
RANDOM_STATE = 42
DATA_DIR = Path("DataTraining")
MODEL_DIR = DATA_DIR / "models"

TRAIN_PATH = DATA_DIR / "water_quality_train_1150.csv"
TEST_PATH = DATA_DIR / "water_quality_test_650.csv"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Training file: {TRAIN_PATH}")
print(f"Testing file: {TEST_PATH}")
print(f"Model output folder: {MODEL_DIR}")


# 2.0 Load Preprocessed Data

This section loads the final train/test CSV files created in `Preprocessing.ipynb`.


## 2.1 Load Training And Testing Files

The files already contain the selected features and the target column `is_safe`.


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(f"Training dataset shape: {train_df.shape}")
print(f"Testing dataset shape: {test_df.shape}")

display(train_df.head())


## 2.2 Separate Features And Target

`X_train` and `X_test` contain the selected water-quality features. `y_train` and `y_test` contain the safety label.


In [ ]:
TARGET_COL = "is_safe"

X_train = train_df.drop(columns=[TARGET_COL])
y_train = train_df[TARGET_COL]

X_test = test_df.drop(columns=[TARGET_COL])
y_test = test_df[TARGET_COL]

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")


## 2.3 Verify Class Distribution

Both training and testing sets should remain balanced after preprocessing.


In [ ]:
print("Training target distribution:")
display(y_train.value_counts())

print("Testing target distribution:")
display(y_test.value_counts())


# 3.0 Model Training Strategy

Three supervised classification algorithms are trained and compared: Random Forest, Decision Tree, and SVM.


## 3.1 Define Cross-Validation Strategy

5-fold stratified cross-validation is used during hyperparameter tuning. Only the training data is used during tuning.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("Cross-validation strategy: 5-fold StratifiedKFold")


## 3.2 Define Evaluation Function

This helper function evaluates each optimized model on the untouched 650-record test set.


In [ ]:
def evaluate_model(model_name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

    tn, fp, fn, tp = cm.ravel()

    result = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision_Safe_Class_1": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "Recall_Safe_Class_1": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "F1_Safe_Class_1": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
        "Recall_Unsafe_Class_0": recall_score(y_test, y_pred, pos_label=0, zero_division=0),
        "False_Safe_Count": int(fp),
        "False_Unsafe_Count": int(fn),
    }

    print(f"\\n{model_name} Confusion Matrix")
    print("Rows = Actual [0 unsafe, 1 safe]; Columns = Predicted [0 unsafe, 1 safe]")
    print(cm)

    print(f"\\n{model_name} Classification Report")
    print(classification_report(y_test, y_pred, target_names=["Unsafe", "Safe"], zero_division=0))

    return result


# 4.0 Random Forest Model

Random Forest combines multiple decision trees and uses voting to make a final classification. It is useful when one Decision Tree is too simple or unstable.


## 4.1 Define Random Forest Hyperparameter Grid

The main Random Forest hyperparameters control the number of trees, tree depth, and minimum samples required for splits/leaves.


In [ ]:
rf_model = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

rf_param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [7, 10, None],
    "min_samples_split": [2, 10],
    "min_samples_leaf": [1, 2],
}

print("Random Forest hyperparameter grid prepared.")


## 4.2 Tune And Train Random Forest

GridSearchCV selects the best Random Forest parameters using 5-fold cross-validation on the training data.


In [ ]:
rf_grid = GridSearchCV(
    estimator=rf_model,
    param_grid=rf_param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
)

rf_grid.fit(X_train, y_train)

print("Best Random Forest parameters:")
print(rf_grid.best_params_)
print(f"Best Random Forest cross-validation F1: {rf_grid.best_score_:.4f}")


## 4.3 Save Random Forest Hyperparameter Results

The cross-validation results are saved for the hyperparameter analysis table.


In [ ]:
rf_results = pd.DataFrame(rf_grid.cv_results_)
rf_results.to_csv(DATA_DIR / "random_forest_hyperparameter_results.csv", index=False)

display(rf_results[["params", "mean_test_score", "std_test_score", "rank_test_score"]].sort_values("rank_test_score").head(10))


# 5.0 Decision Tree Model

Decision Tree does not require scaling. It is also useful for model interpretation through feature importance.


## 5.1 Define Decision Tree Hyperparameter Grid

The main Decision Tree hyperparameters control tree depth and minimum samples required for splits/leaves.


In [ ]:
dt_model = DecisionTreeClassifier(random_state=RANDOM_STATE)

dt_param_grid = {
    "max_depth": [3, 5, 7, 10, 15, 20, None],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 5, 10],
}

print("Decision Tree hyperparameter grid prepared.")


## 5.2 Tune And Train Decision Tree

GridSearchCV selects the best Decision Tree parameters using only the training data.


In [ ]:
dt_grid = GridSearchCV(
    estimator=dt_model,
    param_grid=dt_param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
)

dt_grid.fit(X_train, y_train)

print("Best Decision Tree parameters:")
print(dt_grid.best_params_)
print(f"Best Decision Tree cross-validation F1: {dt_grid.best_score_:.4f}")


## 5.3 Save Decision Tree Hyperparameter Results

The cross-validation results are saved for the hyperparameter analysis table.


In [ ]:
dt_results = pd.DataFrame(dt_grid.cv_results_)
dt_results.to_csv(DATA_DIR / "decision_tree_hyperparameter_results.csv", index=False)

display(dt_results[["params", "mean_test_score", "std_test_score", "rank_test_score"]].sort_values("rank_test_score").head(10))


# 6.0 Support Vector Machine Model

SVM is sensitive to feature scale, so StandardScaler is included in the pipeline.


## 6.1 Define SVM Pipeline And Hyperparameter Grid

The main SVM hyperparameters are kernel, C, and gamma.


In [ ]:
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(random_state=RANDOM_STATE)),
])

svm_param_grid = {
    "svm__kernel": ["linear", "rbf"],
    "svm__C": [0.1, 1, 10, 100],
    "svm__gamma": ["scale", 0.001, 0.01, 0.1, 1],
}

print("SVM hyperparameter grid prepared.")


## 6.2 Tune And Train SVM

GridSearchCV selects the best SVM parameters using only the training data.


In [ ]:
svm_grid = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=svm_param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
)

svm_grid.fit(X_train, y_train)

print("Best SVM parameters:")
print(svm_grid.best_params_)
print(f"Best SVM cross-validation F1: {svm_grid.best_score_:.4f}")


## 6.3 Save SVM Hyperparameter Results

The cross-validation results are saved for the hyperparameter analysis table.


In [ ]:
svm_results = pd.DataFrame(svm_grid.cv_results_)
svm_results.to_csv(DATA_DIR / "svm_hyperparameter_results.csv", index=False)

display(svm_results[["params", "mean_test_score", "std_test_score", "rank_test_score"]].sort_values("rank_test_score").head(10))


# 7.0 Final Model Evaluation

The optimized models are evaluated once on the 650 testing records.


## 7.1 Evaluate All Optimized Models

The test set was not used during feature selection or hyperparameter tuning.


In [ ]:
model_results = []

model_results.append(evaluate_model("Random Forest", rf_grid.best_estimator_, X_test, y_test))
model_results.append(evaluate_model("Decision Tree", dt_grid.best_estimator_, X_test, y_test))
model_results.append(evaluate_model("SVM", svm_grid.best_estimator_, X_test, y_test))


## 7.2 Compare Model Performance

The comparison table summarizes accuracy, precision, recall, F1 score, and false-safe errors.


In [ ]:
comparison_df = pd.DataFrame(model_results)

metric_cols = [
    "Accuracy",
    "Precision_Safe_Class_1",
    "Recall_Safe_Class_1",
    "F1_Safe_Class_1",
    "Recall_Unsafe_Class_0",
]

comparison_display = comparison_df.copy()
comparison_display[metric_cols] = (comparison_display[metric_cols] * 100).round(2)

display(comparison_display.sort_values("F1_Safe_Class_1", ascending=False))


## 7.3 Model Performance Comparison Graph

This graph compares the three classification models using Accuracy, Precision, Recall, and F1-score.


In [ ]:
graph_df = comparison_df[[
    "Model",
    "Accuracy",
    "Precision_Safe_Class_1",
    "Recall_Safe_Class_1",
    "F1_Safe_Class_1",
]].copy()

graph_df = graph_df.rename(columns={
    "Precision_Safe_Class_1": "Precision",
    "Recall_Safe_Class_1": "Recall",
    "F1_Safe_Class_1": "F1 Score",
})

graph_df = graph_df.melt(
    id_vars="Model",
    var_name="Metric",
    value_name="Metric Value",
)
graph_df["Metric Value"] = graph_df["Metric Value"] * 100

plt.figure(figsize=(10, 6))
sns.barplot(data=graph_df, x="Model", y="Metric Value", hue="Metric")
plt.title("Classification Model Performance Comparison")
plt.ylabel("Metric Value (%)")
plt.xlabel("Classification Model")
plt.ylim(0, 100)
plt.legend(title="Evaluation Metric")
plt.tight_layout()
plt.show()


# 8.0 Save Models And Results

This section saves the optimized models and final comparison results.


## 8.1 Save Trained Models

Each saved model includes its preprocessing pipeline if scaling is required.


In [ ]:
joblib.dump(rf_grid.best_estimator_, MODEL_DIR / "random_forest_model.pkl")
joblib.dump(dt_grid.best_estimator_, MODEL_DIR / "decision_tree_model.pkl")
joblib.dump(svm_grid.best_estimator_, MODEL_DIR / "svm_model.pkl")

print("Models saved successfully.")


## 8.2 Save Final Comparison Results

The final test-set model comparison is saved as a CSV file.


In [ ]:
comparison_df.to_csv(DATA_DIR / "model_comparison_results.csv", index=False)

print("Saved model comparison results to:")
print(DATA_DIR / "model_comparison_results.csv")
